this notebook contains the multivariate analysis of the incident types for the isochrone approach

# 1.0 Load packages

In [ ]:
# load the packages
import pandas as pd
import json
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from scipy.stats import chi2_contingency, fisher_exact
from statsmodels.stats.multitest import multipletests
from statsmodels.discrete.discrete_model import NegativeBinomial
import warnings

In [ ]:
# suppress RuntimeWarnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [ ]:
import gc
gc.collect()

In [ ]:
# import boundary data
bradford_boundary = gpd.read_file('bradford_boundary.geojson')

In [ ]:
# load the data
df = gpd.read_file('incident_isochrones.geojson', use_arrow=True)

In [ ]:
df.columns

# 2,0 Data separation

In [ ]:
# filter for each time category
crimes_df1 = df[df['trip_time_min'] <= 5].copy()
crimes_df2 = df[(df['trip_time_min'] > 5) | (df['trip_time_min'].isna())].copy()
crimes_df3 = df[df['trip_time_min'] <= 10].copy()
crimes_df4 = df[(df['trip_time_min'] > 10) | (df['trip_time_min'].isna())].copy()
crimes_df5 = df[df['trip_time_min'] <= 15].copy()
crimes_df6 = df[(df['trip_time_min'] > 15) | (df['trip_time_min'].isna())].copy()
crimes_df7 = df[df['trip_time_min'] <= 20].copy()
crimes_df8 = df[(df['trip_time_min'] > 20) | (df['trip_time_min'].isna())].copy()
crimes_df9 = df[df['trip_time_min'] <= 25].copy()
crimes_df10 = df[(df['trip_time_min'] > 25) | (df['trip_time_min'].isna())].copy()
crimes_df11 = df[df['trip_time_min'] <= 30].copy()
crimes_df12 = df[(df['trip_time_min'] > 30) | (df['trip_time_min'].isna())].copy()

In [ ]:
# track the buffer distance
crimes_df1['time'] = '5'
crimes_df2['time'] = '5+'
crimes_df3['time'] = '10'
crimes_df4['time'] = '10+'
crimes_df5['time'] = '15'
crimes_df6['time'] = '15+'
crimes_df7['time'] = '20'
crimes_df8['time'] = '20+'
crimes_df9['time'] = '25'
crimes_df10['time'] = '25+'
crimes_df11['time'] = '30'
crimes_df12['time'] = '30+'

In [ ]:
# concatenante the incidentypes
crimes = pd.concat(
    [crimes_df1, crimes_df2, crimes_df3, crimes_df4,
     crimes_df5, crimes_df6, crimes_df7, crimes_df8,
    crimes_df9, crimes_df10, crimes_df11, crimes_df12],
    ignore_index=True)

The adjusted statistical test had identified on violence without injury and public order offences.

In [ ]:
sig_crimes = crimes[crimes['incident'].isin([
    'abandoned / no information', 'ccc use only - info log',
    'concern for safety', 'emergency service hoax', 'harassment',
    'highway disruption', 'ms intruder - false alarm', 'nuisance',
    'public order', 'rtc - damage only - det exch nfa',
    'rtc - slight injury - det exch nfa', 'theft - shoplifting',
    'violence against person'])]

In [ ]:
sig_crimes.head()

# 3.0 Model selection

In [ ]:
# pivot teh table
wide_counts = pd.pivot_table(
    data=sig_crimes,
    index=['year&week', 'term_time_type', 'time'],
    columns='incident',
    values='season',
    aggfunc='count',
    fill_value=0
)

In [ ]:
# compute mean and variance
# loop through buffer
all_buffer_results = []
unique_buffers = sig_crimes['time'].unique()

for current_buffer in unique_buffers:
    # Isolate the rows belonging to the current buffer zone
    buffer_subset = wide_counts.xs(current_buffer, level='time')
    
    # Calculate localized mean and variance across the timeline rows
    buffer_means = buffer_subset.mean()
    buffer_variances = buffer_subset.var()
    
    # Compile the results for this specific distance band
    buffer_df = pd.DataFrame({
        'crime_type': buffer_means.index,
        'mean_count': buffer_means.values,
        'variance': buffer_variances.values,
        'time': current_buffer
    })
    
    all_buffer_results.append(buffer_df)

In [ ]:
# put into a table
dispersion = pd.concat(all_buffer_results, axis=0)

In [ ]:
# calculate teh ratio as a way to identify dispersion
dispersion['dispersion_ratio'] = dispersion['variance'] / dispersion['mean_count']
dispersion = dispersion.sort_values(by='dispersion_ratio', ascending=False)

In [ ]:
pd.set_option('display.max_rows', None)
dispersion

I will only use the Negative Binomial model

## 3.1 Build the table

In [ ]:
# get the dates
calendar = (
    sig_crimes[['date_only', 'term_time_type']]
    .drop_duplicates())

In [ ]:
# get the crimes and the buffer
crime_table = pd.DataFrame({
    'incident': sig_crimes['incident'].unique()})

buffer_table = pd.DataFrame({
    'time': sig_crimes['time'].unique()})

In [ ]:
# merge the crimes and distances to the daily
sig_df = (
    calendar.merge(crime_table, how='cross')
    .merge(buffer_table, how='cross'))

In [ ]:
# count crimes at each day and buffer
observed_counts = (
    sig_crimes
    .groupby(['date_only', 'incident', 'term_time_type', 'time'])
    .size()
    .reset_index(name='crime_count'))

In [ ]:
# rename buffer to fit teh sig_df table
observed_counts = observed_counts.rename(
    columns={'time': 'time'}
)

In [ ]:
# add the counts
sig_df = sig_df.merge(
    observed_counts,
    on=['date_only', 'incident', 'term_time_type', 'time'],
    how='left')

In [ ]:
# fill no crime with 0
sig_df['crime_count'] = (
    sig_df['crime_count']
    .fillna(0)
    .astype(int)
)

In [ ]:
# encode day and distance
sig_df['is_school_day'] = sig_df['term_time_type'].map({
    'Term time': 1,
    'holiday': 0
})

sig_df['is_time_within'] = (
    sig_df['time']
    .str.endswith('+')
    .map({True: 1,
          False: 0})
)

# 4.0 Negative Binomial Regression

Daily crime counts are modelled using Negative Binomial regression. This specification is appropriate for count outcomes and accommodates overdispersion in the daily crime-count distribution. The model includes proximity to a secondary school, school-day status, and their interaction. Cluster-robust standard errors are estimated by date to account for dependence between proximity groups observed on the same day.

he model examines whether crime counts differ according to:

proximity to a secondary school;
whether the date is a school day or holiday; and
the interaction between proximity and school-day status.

The count model is:

$$ \log(\mu_i) = \beta_0 + \beta_1\text{Within}_i + \beta_2\text{SchoolDay}_i + \beta_3(\text{Within}_i\times\text{SchoolDay}_i) $$

where:

$$ \mu_i = E(Y_i|X_i) $$

and:

\(Y_i\) = observed daily crime count for observation \(i\)
\(\mu_i\) = expected daily crime count
\(\beta_0\) = intercept: expected log crime count for the reference group
\(\beta_1\) = effect of being within the specified proximity threshold rather than further away
\(\beta_2\) = effect of a school day rather than a holiday
\(\beta_3\) = interaction effect, indicating whether the within-versus-further relationship changes on school days
Within = 1 for observations within the specified buffer and 0 for further away
SchoolDay = 1 for school days and 0 for holidays

The reference category is therefore:

Further away + Holiday

In [ ]:
# model formula
formula = 'crime_count ~ C(is_time_within) * is_school_day'

In [ ]:
# get the offences and buffer distances
incidents= sorted(
    sig_df['incident']
    .dropna()
    .unique())

# Each number identifies one distance pair
times = [5, 10, 15, 20, 25, 30]

In [ ]:
# Store models by offence, then time pair
incident_models = {}

for incident in incidents:
    incident_models[incident] = {}

    for minute in times:
        # Select both sides of the pair
        pair = [str(minute), f"{minute}+"]

        model_data = sig_df[
            (sig_df["incident"] == incident) &
            (sig_df["time"].astype(str).isin(pair))
        ].copy()

        # Fit and store the model for this complete pair
        model = NegativeBinomial.from_formula(formula, data=model_data)
        result = model.fit(method="bfgs", maxiter=1000, disp=False, cov_type="cluster",
                           cov_kwds={"groups": model_data["date_only"]})
        
        incident_models[incident][minute] = result

        print(f"\nIncident: {incident} | Time pair: {minute} and {minute}+")
        print(result.summary())

ccc use only - info log 5 vs 5+ (p < 0.05) \
harrasment 5 vs 5+ (p < 0.05) \
highway disruption 20 vs 20+ (p < 0.05) \
ms intruder - false alarm 10 vs 10+, 20 vs 20+(p < 0.05) \
public order 5 vs 5+, 10 vs 10+, 15 vs 15+, 20 vs 20+, 25 vs 25+, 30 vs 30+ (p < 0.05)

In [ ]:
# define interest
terms = {
    "C(is_time_within)[T.1]": "within time coeff",
    "is_school_day": "school day coeff",
    "C(is_time_within)[T.1]:is_school_day": "interaction coeff"
}

In [ ]:
# initialise list
result_rows = []

for incident, time_models in incident_models.items():
    for minute, result in time_models.items():
        for model_term, label in terms.items():
        
            # Extract this term's coefficient and confidence interval
            coefficient = result.params[model_term]
            lower, upper = result.conf_int().loc[model_term]

            result_rows.append({
                "incident": incident,
                "time": minute,
                "term": label,
                "coefficient": coefficient,
                "irr": np.exp(coefficient),
                "irr_ci_lower": np.exp(lower),
                "irr_ci_upper": np.exp(upper),
                "p_value": result.pvalues[model_term],
                "n_observations": result.nobs
            })

In [ ]:
# put into a datframe and sort
model_results_df = (
    pd.DataFrame(result_rows)
    .sort_values(["incident", "time"])
    .reset_index(drop=True)
)

# model_results_df.round(4)

In [ ]:
# save model results
model_results_df.to_csv('isochrone_incidenttype_model_results.csv')

## 4.1 Model evaluation

In [ ]:
# initialise list
evaluation = []

for incident, time_models in incident_models.items():
    for minute, result in time_models.items():
        # Observed and fitted daily crime counts for rows used by this model
        observed = np.asarray(result.model.endog)
        predicted = np.asarray(result.predict())

        # NB2 variance: mean + alpha × mean²
        alpha = result.params["alpha"]
        variance = predicted + alpha * predicted**2

        # Pearson dispersion uses the observation-level residuals
        pearson_residuals = (observed - predicted) / np.sqrt(variance)
        pearson_dispersion = (
            np.sum(pearson_residuals**2) / result.df_resid
        )

        # Probability of zero counts under the fitted NB2 model
        predicted_zero_percent = (
            (1 + alpha * predicted) ** (-1 / alpha)
        ).mean() * 100

        evaluation.append({
            "incident": incident,
            "time_min": minute,
            "n_observations": result.nobs,
            "converged": result.mle_retvals["converged"],
            "alpha": alpha,
            "pearson_dispersion": pearson_dispersion,
            "mae": np.mean(np.abs(observed - predicted)),
            "rmse": np.sqrt(np.mean((observed - predicted)**2)),
            "observed_zero_percent": np.mean(observed == 0) * 100,
            "predicted_zero_percent": predicted_zero_percent
        })

In [ ]:
# put into a dataframe
evaluation_df = (
    pd.DataFrame(evaluation)
    .sort_values(["incident", "time_min"])
    .reset_index(drop=True)
)

display(evaluation_df)

All models converged, Pearson dispersion statistics remained close to one, and predicted zero proportions closely matched observed values. Public order offence models demonstrated slightly stronger predictive performance, with lower MAE and RMSE values. Violence without injury exhibited higher prediction errors but still showed acceptable dispersion and accurate reproduction of zero counts. The results suggest that the fitted negative binomial models provide an adequate representation of crime counts across the 5-30 minute isochrone bands.

In [ ]:
# save result
evaluation_df.to_csv('isochrone_incidenttype_evaluation_results.csv')